# Customer Churn — Responsible Baseline Notebook

This notebook audits the supplied 12-row training dataset and evaluates transparent baselines before any production model is considered.

**Important:** The dataset is too small for production claims. The notebook is a reproducible demonstration of the requested responsible-ML workflow.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

df = pd.read_csv("customer-churn-training.csv")
df

## 1. Data quality checks

The supplied data card template asks for missingness, duplicates, outliers, class balance, and train/test separation. The source file has no timestamps or split indicator, so temporal separation cannot be verified.


In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

print("\nTarget counts:")
print(df["churned"].value_counts().sort_index())

print("\nNumeric summary:")
print(df[["tenure_months","support_tickets","monthly_spend_inr","last_login_days"]].describe())

In [ ]:
# IQR-based outlier screen (descriptive only; n=12 is very small)
numeric_cols = ["tenure_months","support_tickets","monthly_spend_inr","last_login_days"]
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    flagged = df.loc[(df[col] < lo) | (df[col] > hi), ["customer_id", col]]
    print(f"{col}: range=({df[col].min()}, {df[col].max()}), IQR bounds=({lo:.2f}, {hi:.2f}), flagged={len(flagged)}")

## 2. Baseline A — majority class

The majority class is `churned = 0`. This is a useful floor but it cannot identify any churners.


In [ ]:
y = df["churned"]
majority_pred = np.zeros(len(df), dtype=int)

print("Accuracy:", accuracy_score(y, majority_pred))
print("Precision:", precision_score(y, majority_pred, zero_division=0))
print("Recall:", recall_score(y, majority_pred, zero_division=0))
print("F1:", f1_score(y, majority_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y, majority_pred))

## 3. Baseline B — transparent human-review rule

Rule:

`flag = last_login_days >= 14 OR support_tickets >= 4`

This is an operational baseline, not a learned model.


In [ ]:
rule_pred = (
    (df["last_login_days"] >= 14) |
    (df["support_tickets"] >= 4)
).astype(int)

cm = confusion_matrix(y, rule_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix:\n", cm)
print("Accuracy:", accuracy_score(y, rule_pred))
print("Precision:", precision_score(y, rule_pred, zero_division=0))
print("Recall:", recall_score(y, rule_pred, zero_division=0))
print("F1:", f1_score(y, rule_pred, zero_division=0))
print("TN, FP, FN, TP:", tn, fp, fn, tp)

## 4. Error-cost view

Initial working assumption: false negative = 5 cost units; false positive = 1 cost unit. These costs must be validated with the business owner.

`cost = 5*FN + 1*FP`


In [ ]:
FN_COST = 5
FP_COST = 1
cost = FN_COST * fn + FP_COST * fp
print("Illustrative cost:", cost)

## 5. Leakage and representation warnings

- `customer_id` is an identifier and is excluded from modeling.
- No timestamp is available, so temporal leakage cannot be ruled out.
- `plan_type` perfectly separates the target in this sample: all Basic records churn and no Standard/Pro records churn.
- The sample contains only 12 records; random train/test metrics would be highly unstable.
- A production evaluation should use a larger, time-separated dataset.


## 6. Optional demonstration model

If a learned model is tested, use an interpretable pipeline and treat results as exploratory. Do not present the tiny-sample score as production performance.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict

X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

num_cols = ["tenure_months","support_tickets","monthly_spend_inr","last_login_days"]
cat_cols = ["plan_type"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

model = Pipeline([
    ("preprocess", pre),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Leave-one-out is used only to avoid choosing a tiny arbitrary train/test split.
loo = LeaveOneOut()
proba = cross_val_predict(model, X, y, cv=loo, method="predict_proba")[:, 1]
pred = (proba >= 0.5).astype(int)

print("LOO accuracy:", accuracy_score(y, pred))
print("LOO precision:", precision_score(y, pred, zero_division=0))
print("LOO recall:", recall_score(y, pred, zero_division=0))
print("LOO F1:", f1_score(y, pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y, pred))

# Note: these estimates remain extremely uncertain because n=12.


## 7. Deployment checklist

Before deployment:
1. Document provenance, consent/licensing, and intended population.
2. Add a scoring timestamp and define the future churn window.
3. Verify every feature is available before the scoring cutoff.
4. Build a time-based train/validation/test split.
5. Validate false-positive and false-negative costs.
6. Check calibration and subgroup error rates where legitimate group data exists.
7. Add abstention/human-review rules.
8. Define monitoring and rollback owners.
